In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


# **1. ANSWERS**
1. We are trying to build a model that will predict the sale price of houses in Ames, Iowa.
2. This a supervised learning tasks since we are using labelled data. Since our target variable is the "SalePrice", which is a continous value, this is specifcally a regression problem.
3. The competition has specfically stated that performance should be measured in Root Mean Squared Logarithmic Error (RMSLE).

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
%%bash
cat /kaggle/input/house-prices-advanced-regression-techniques/data_description.txt

In [ ]:
train_df = pd.read_csv(
    "/kaggle/input/house-prices-advanced-regression-techniques/train.csv",
    index_col="Id",
)

test_df = pd.read_csv(
    "/kaggle/input/house-prices-advanced-regression-techniques/test.csv",
    index_col="Id",
)

In [ ]:
train_df.info()

In [ ]:
print("Train shape:", train_df.shape)

In [ ]:
print("\nTrain dtypes:")
print(train_df.dtypes.value_counts())

In [ ]:
num_cols_train = train_df.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols_train = train_df.select_dtypes(include=["object"]).columns.tolist()
print(f"\nTrain numeric cols: {len(num_cols_train)} | categorical cols: {len(cat_cols_train)}")

In [ ]:
display(train_df.head(10))

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

1. We have 43 object, 34 int64, and 3 float types of features. 28 features were categorical, most of which were object data types, but I also labelled 3 int64 types as categorical, which are the "MSSubClass", "BsmtFinSF1", "BsmtFinSF2", as the numbers represent a category in that case.

13 objects were ordinal (ExterQual, ExterCond, etc...) Two int64 data types were specfically bounded, which are the "OverallQual", "OverallCond".

31 features were unbounded numeric features, including all float64 data types.

I would like to introduce a "Time" based numeric type, such as year and month, since they are technically unbounded, but also can be used in different ways. I have 5 found features of this type. (YearBuilt, YearRemodAdd, GarageYrBlt, YrSold, MoSold) Features can be enginereed to be: “HouseAge = YrSold - YearBuilt” and “YearsSinceRemodel = YrSold - YearRemodAdd”

Our target variable is a continous (unbounded) numeric (int64) {SalePrice}.

2. 19 features have missing values, which are:



In [ ]:
missing_features = train_df.isnull().sum()
missing_features = missing_features[missing_features > 0]

print(f"Number of features with missing values: {len(missing_features)}\n")
print(missing_features.sort_values(ascending=False))

3. To calculate the outliers for numeric values, their is a statiscal way we can do it using the Z-score, where we can calculate outlier values that are 3 standard deviations away from the mean.

In [ ]:
from scipy import stats
import numpy as np

z_scores = np.abs(stats.zscore(train_df.select_dtypes(include=[np.number])))
outliers = (z_scores > 3).sum(axis=0)
outliers[outliers > 0].sort_values(ascending=False).head(10)


In [ ]:
cols = [
    "BsmtHalfBath",
    "KitchenAbvGr",
    "ScreenPorch",
    "EnclosedPorch",
    "BsmtFinSF2",
    "MSSubClass",
    "OverallCond",
    "OpenPorchSF",
    "3SsnPorch",
    "WoodDeckSF"
]

for col in cols:
    print(f"\n--- {col} ---")
    print(train_df[col].value_counts().sort_index())


In [ ]:
numeric_cols = train_df.select_dtypes(include=["int64","float64"]).columns
train_df[numeric_cols].plot(kind="box", subplots=True, layout=(10,4), figsize=(15,30), sharex=False)
plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(x="BsmtHalfBath", data=train_df)
plt.title("Distribution of Basement Half Baths")
plt.show()

In [ ]:
train_df["BsmtHalfBath"].value_counts().sort_index()

As for objects, we can show text, or categories that occur in less than 1% of the rows.

In [ ]:
cat_cols = train_df.select_dtypes(include="object").columns

threshold = 0.01 * len(train_df)

for col in cat_cols:
    counts = train_df[col].value_counts(dropna=False)
    rare = counts[counts < threshold]
    if not rare.empty:
        print(f"\n--- {col} ---")
        print(rare)


In [ ]:
train_df["Street"].value_counts(dropna=False)


4. In general, useful features are features.Useless features can be classfied as features that are constant, redundant, or rarely populated. Features like "Kitchen Above Ground" where they are almost always at the same value (1), can be dropped, as they will not benefit in identifying a correltaion with our target prediction (Sale Price). On the other hand useful features will be features that appear to have a strong relationship with the target prediction, and that are varied enough to be able to make out a correlation if possible.

Based on the histograms and the bar graph printed, I would drop features such as "BsmtHalfBath" "Utilities", and "Street".

In [ ]:
# Select numeric columns only
num_cols = train_df.select_dtypes(include=["int64", "float64"]).columns

# Plot histograms
train_df[num_cols].hist(figsize=(20, 20), bins=30, edgecolor='black')
plt.suptitle("Histograms of All Numeric Features", fontsize=18)
plt.show()


In [ ]:
import seaborn as sns

cat_cols = train_df.select_dtypes(include="object").columns

for col in cat_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(y=col, data=train_df, order=train_df[col].value_counts().index)
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()


5. **Identify and characterize the target distribution.**

Distribution is right-skewed, meaning that most homes are moderately priced, with a few expensive outliers.
The histogram shows a long right tail, very few high-priced homes.
This non-normal shape can reduce regression performance.
We will do a log transformation to normalize it.

In [ ]:
sns.histplot(train_df["SalePrice"], bins=40, kde=True)
plt.title("Distribution of SalePrice")
plt.xlabel("Sale Price (USD)")
plt.ylabel("Frequency")
plt.show()

6. **Study the correlations between attributes.**

In general, these are our findings:
OverallQual	0.79	The single most important predictor. Better construction quality and materials means higher price.
GrLivArea	0.71	Larger above-ground living area strongly increases price.
GarageCars	0.64	More garage capacity means higher price. Indicates home size and utility.
GarageArea	0.62	Larger garage area correlates with higher value (related to GarageCars).
TotalBsmtSF	0.61	Bigger basements often add usable space and value.

In [ ]:
# Compute correlation matrix (numeric features only)
corr_matrix = train_df.corr(numeric_only=True)

# Display overall correlation with target
corr_with_target = corr_matrix["SalePrice"].sort_values(ascending=False)
print(corr_with_target.head(15))

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(corr_matrix, cmap='coolwarm', vmax=0.8, vmin=-0.3)
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

In [ ]:
top_features = corr_with_target.index[1:11]  # top 10 features excluding SalePrice
plt.figure(figsize=(8,6))
sns.heatmap(train_df[top_features].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Top 10 Correlated Features with SalePrice")
plt.show()

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

# Feature selection **Version 1**

In [ ]:
from scipy import stats
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import RidgeCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.compose import TransformedTargetRegressor

RANDOM_STATE = 42

# --- Load ---
train_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv", index_col="Id")
test_df  = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv",  index_col="Id")

y = train_df["SalePrice"].copy()
X = train_df.drop(columns=["SalePrice"]).copy()
X_test = test_df.copy()

# --- 1) Drop any feature that has missing values (in train OR test) ---
na_cols_train = X.columns[X.isnull().any()]
na_cols_test  = X_test.columns[X_test.isnull().any()]
na_cols_union = sorted(set(na_cols_train).union(set(na_cols_test)))

X.drop(columns=na_cols_union, inplace=True)
X_test.drop(columns=na_cols_union, inplace=True)

# --- 2) Drop numeric features that have any outliers (|z| > 3) in TRAIN ---
num_cols = X.select_dtypes(include=[np.number]).columns
# Compute z-scores on train numeric columns only
z = pd.DataFrame(stats.zscore(X[num_cols], nan_policy='omit'), index=X.index, columns=num_cols)
# Columns with any absolute z > 3
outlier_cols = sorted([c for c in num_cols if np.nanmax(np.abs(z[c].values)) > 3])

X.drop(columns=outlier_cols, inplace=True)
X_test.drop(columns=[c for c in outlier_cols if c in X_test.columns], inplace=True)

# --- 3) Create engineered features (ensure inputs exist first) ---
def add_age_features(df):
    df = df.copy()
    if set(["YrSold","YearBuilt"]).issubset(df.columns):
        df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    else:
        df["HouseAge"] = 0  # fallback if columns were dropped
    if set(["YrSold","YearRemodAdd"]).issubset(df.columns):
        df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]
    else:
        df["YearsSinceRemodel"] = 0
    # Optional: clip negatives (in case of data anomalies)
    df["HouseAge"] = df["HouseAge"].clip(lower=0)
    df["YearsSinceRemodel"] = df["YearsSinceRemodel"].clip(lower=0)
    return df

X = add_age_features(X)
X_test = add_age_features(X_test)

# --- 4) One-hot encode all remaining categorical columns ---
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

In [ ]:
# Select numeric columns only
num_cols = train_df.select_dtypes(include=["int64", "float64"]).columns

# Plot histograms
train_df[num_cols].hist(figsize=(20, 20), bins=30, edgecolor='black')
plt.suptitle("Histograms of All Numeric Features", fontsize=18)
plt.show()

report = {}

# 1) Basic shapes after your drops + feature engineering (pre-OHE)
report["n_samples_train"] = X.shape[0]
report["n_samples_test"]  = X_test.shape[0]
report["n_features_pre_ohe_train"] = X.shape[1]
report["n_features_pre_ohe_test"]  = X_test.shape[1]

# 2) Dropped columns counts
report["dropped_due_to_missing"] = len(na_cols_union)
report["dropped_due_to_outliers"] = len(outlier_cols)

# 3) Engineered features added
engineered_cols = ["HouseAge", "YearsSinceRemodel"]
report["engineered_features_present"] = sum(c in X.columns for c in engineered_cols)

# 4) One-hot encoding feature counts (fit on train)
if len(cat_cols) > 0:
    _enc = OneHotEncoder(handle_unknown="ignore")
    _enc.fit(X[cat_cols])
    ohe_per_col = {col: len(cats) for col, cats in zip(cat_cols, _enc.categories_)}
    report["ohe_features_total"] = int(sum(ohe_per_col.values()))
    report["ohe_features_by_column"] = ohe_per_col
else:
    report["ohe_features_total"] = 0
    report["ohe_features_by_column"] = {}

# 5) Total features AFTER preprocess (OHE + numeric passthrough)
preprocess_fitted = preprocess.fit(X)
feat_names = preprocess_fitted.get_feature_names_out()
report["n_features_post_ohe_train"] = len(feat_names)

# Pretty print summary
print("=== Dataset Shape (after drops & engineering, before OHE) ===")
print(f"Train: {report['n_samples_train']} rows, {report['n_features_pre_ohe_train']} columns")
print(f"Test : {report['n_samples_test']} rows, {report['n_features_pre_ohe_test']} columns\n")

print("=== Dropped Columns ===")
print(f"Due to missing values: {report['dropped_due_to_missing']}")
print(f"Due to outliers      : {report['dropped_due_to_outliers']}\n")

print("=== Engineered Features ===")
print(f"Added (present now): {report['engineered_features_present']}  -> {engineered_cols}\n")

print("=== One-Hot Encoding ===")
print(f"Total OHE features added: {report['ohe_features_total']}")
if report["ohe_features_by_column"]:
    # Show top 10 by size to avoid huge printouts
    by_size = sorted(report["ohe_features_by_column"].items(), key=lambda kv: kv[1], reverse=True)
    preview = by_size[:10]
    print("Top 10 categorical columns by number of OHE columns:")
    for col, k in preview:
        print(f"  {col}: {k}")
    if len(by_size) > 10:
        print(f"... ({len(by_size) - 10} more)\n")
else:
    print("(No categorical columns to encode)\n")

print("=== Final Feature Count (after preprocess) ===")
print(f"Train post-OHE features: {report['n_features_post_ohe_train']}")

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


**Unregularized linear model**

In [ ]:
# ===== Train unregularized Linear Regression (default params) on log target =====
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 5) Model on log target; evaluate with log-RMSE ---
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

# Unregularized linear model
base_model = LinearRegression()

# Wrap to learn on log1p(SalePrice)
model = TransformedTargetRegressor(
    regressor=Pipeline([("prep", preprocess), ("linreg", base_model)]),
    func=np.log1p,
    inverse_func=np.expm1,
)

# CV on log-space predictions
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def log_rmse_cv(estimator, X_cv, y_cv):
    scores = cross_val_score(
        estimator,
        X_cv,
        y_cv,
        scoring=make_scorer(
            lambda yt, yp: rmse(np.log1p(yt), np.log1p(yp)),
            greater_is_better=False,
        ),
        cv=kf,
        n_jobs=None
    )
    return -scores  # invert sign since greater_is_better=False

cv_scores = log_rmse_cv(model, X, y)
print(f"5-fold log-RMSE: mean={cv_scores.mean():.5f}  std={cv_scores.std():.5f}")

# --- 6) Fit on full train & predict test ---
model.fit(X, y)
test_pred = model.predict(X_test)

# Sanity: predictions must be positive
test_pred = np.maximum(test_pred, 0)

# --- 7) Save submission ---
sub = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv", index_col="Id")
sub["SalePrice"] = test_pred
sub.to_csv("/kaggle/working/submission.csv")
print("Wrote /kaggle/working/submission.csv")

# --- 8) Log what we dropped for traceability ---
print("\nDropped due to missing values:", len(na_cols_union), na_cols_union[:10], "..." if len(na_cols_union) > 10 else "")
print("Dropped due to outliers:", len(outlier_cols), outlier_cols[:10], "..." if len(outlier_cols) > 10 else "")
print("Remaining shape (train):", X.shape, "| (test):", X_test.shape)

**Regularized linear regression model (Ridge Regression)**

In [ ]:
# ===== Train Ridge (default params) on log target =====
from sklearn.linear_model import Ridge

# --- 5) Model on log target; evaluate with log-RMSE ---
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

# Regularized linear model with default parameters (alpha=1.0)
base_model = Ridge(random_state=RANDOM_STATE)

# Wrap to learn on log1p(SalePrice)
model = TransformedTargetRegressor(
    regressor=Pipeline([("prep", preprocess), ("ridge", base_model)]),
    func=np.log1p,
    inverse_func=np.expm1,
)

# CV on log-space predictions
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def log_rmse_cv(estimator, X_cv, y_cv):
    scores = cross_val_score(
        estimator,
        X_cv,
        y_cv,
        scoring=make_scorer(
            lambda yt, yp: rmse(np.log1p(yt), np.log1p(yp)),
            greater_is_better=False,
        ),
        cv=kf,
        n_jobs=None
    )
    return -scores  # invert sign since greater_is_better=False

cv_scores = log_rmse_cv(model, X, y)
print(f"5-fold log-RMSE: mean={cv_scores.mean():.5f}  std={cv_scores.std():.5f}")

# --- 6) Fit on full train & predict test ---
model.fit(X, y)
test_pred = model.predict(X_test)

# Sanity: predictions must be positive
test_pred = np.maximum(test_pred, 0)

# --- 7) Save submission ---
sub = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv", index_col="Id")
sub["SalePrice"] = test_pred
sub.to_csv("/kaggle/working/submission.csv")
print("Wrote /kaggle/working/submission.csv")

# --- 8) Log what we dropped for traceability ---
print("\nDropped due to missing values:", len(na_cols_union), na_cols_union[:10], "..." if len(na_cols_union) > 10 else "")
print("Dropped due to outliers:", len(outlier_cols), outlier_cols[:10], "..." if len(outlier_cols) > 10 else "")
print("Remaining shape (train):", X.shape, "| (test):", X_test.shape)


Decision Tree

In [ ]:
# ===== Train Decision Tree Regressor on log target =====
from sklearn.tree import DecisionTreeRegressor

# --- 5) Model on log target; evaluate with log-RMSE ---
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

# Plain decision tree (defaults); add random_state for reproducibility
base_model = DecisionTreeRegressor(random_state=RANDOM_STATE)

# Wrap to learn on log1p(SalePrice)
model = TransformedTargetRegressor(
    regressor=Pipeline([("prep", preprocess), ("tree", base_model)]),
    func=np.log1p,
    inverse_func=np.expm1,
)

# CV on log-space predictions
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def log_rmse_cv(estimator, X_cv, y_cv):
    scores = cross_val_score(
        estimator,
        X_cv,
        y_cv,
        scoring=make_scorer(
            lambda yt, yp: rmse(np.log1p(yt), np.log1p(yp)),
            greater_is_better=False,
        ),
        cv=kf,
        n_jobs=None
    )
    return -scores  # invert sign since greater_is_better=False

cv_scores = log_rmse_cv(model, X, y)
print(f"5-fold log-RMSE: mean={cv_scores.mean():.5f}  std={cv_scores.std():.5f}")

# --- 6) Fit on full train & predict test ---
model.fit(X, y)
test_pred = model.predict(X_test)

# Sanity: predictions must be positive
test_pred = np.maximum(test_pred, 0)

# --- 7) Save submission ---
sub = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv", index_col="Id")
sub["SalePrice"] = test_pred
sub.to_csv("/kaggle/working/submission.csv")
print("Wrote /kaggle/working/submission.csv")

# --- 8) Log what we dropped for traceability ---
print("\nDropped due to missing values:", len(na_cols_union), na_cols_union[:10], "..." if len(na_cols_union) > 10 else "")
print("Dropped due to outliers:", len(outlier_cols), outlier_cols[:10], "..." if len(outlier_cols) > 10 else "")
print("Remaining shape (train):", X.shape, "| (test):", X_test.shape)

In [ ]:
# ===== Visualizing overfitting: learning curves & validation curves (log-RMSE) =====
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, learning_curve, validation_curve, cross_validate
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Our Kaggle-style scorer: RMSE on log1p scale (lower is better)
log_rmse_scorer = make_scorer(
    lambda yt, yp: mean_squared_error(np.log1p(yt), np.log1p(yp), squared=False),
    greater_is_better=False
)

def wrap_model(estimator):
    """Attach your preprocess and log-transform the target."""
    return TransformedTargetRegressor(
        regressor=Pipeline([("prep", preprocess), ("est", estimator)]),
        func=np.log1p,
        inverse_func=np.expm1,
    )

def plot_learning_curve_for(model, X, y, title):
    train_sizes = np.linspace(0.1, 1.0, 8)
    tr_sizes, tr_scores, val_scores = learning_curve(
        model, X, y, cv=kf, scoring=log_rmse_scorer, train_sizes=train_sizes, n_jobs=None, shuffle=True, random_state=RANDOM_STATE
    )
    # learning_curve returns negative scores because greater_is_better=False; flip to positive
    tr_mean = -tr_scores.mean(axis=1)
    tr_std  =  tr_scores.std(axis=1)
    val_mean = -val_scores.mean(axis=1)
    val_std  =  val_scores.std(axis=1)

    plt.figure(figsize=(7,5))
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("log-RMSE")
    plt.grid(True)

    # Train
    plt.plot(tr_sizes, tr_mean, label="Training score")
    plt.fill_between(tr_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.1)

    # CV
    plt.plot(tr_sizes, val_mean, label="CV score")
    plt.fill_between(tr_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1)

    plt.legend()
    plt.show()

def plot_validation_curve_for(model_name, estimator, param_name, param_range, X, y):
    """Plot validation curve: model complexity vs log-RMSE (train & CV)."""
    model = wrap_model(estimator)
    train_scores, val_scores = validation_curve(
        model, X, y,
        param_name=f"regressor__est__{param_name}",
        param_range=param_range,
        cv=kf,
        scoring=log_rmse_scorer,
        n_jobs=None
    )
    tr_mean = -train_scores.mean(axis=1)
    tr_std  =  train_scores.std(axis=1)
    val_mean = -val_scores.mean(axis=1)
    val_std  =  val_scores.std(axis=1)

    plt.figure(figsize=(7,5))
    plt.title(f"{model_name} validation curve: {param_name}")
    plt.xlabel(param_name)
    plt.ylabel("log-RMSE")
    plt.grid(True)

    # If param_range is logspace, semi-log x-axis is helpful
    if np.all(param_range > 0) and (np.max(param_range)/np.min(param_range) > 50):
        plt.xscale("log")

    plt.plot(param_range, tr_mean, label="Training score")
    plt.fill_between(param_range, tr_mean - tr_std, tr_mean + tr_std, alpha=0.1)

    plt.plot(param_range, val_mean, label="CV score")
    plt.fill_between(param_range, val_mean - val_std, val_mean + val_std, alpha=0.1)

    plt.legend()
    plt.show()

# 1) Learning curves for three models
lin_model   = wrap_model(LinearRegression())
ridge_model = wrap_model(Ridge(random_state=RANDOM_STATE))
tree_model  = wrap_model(DecisionTreeRegressor(random_state=RANDOM_STATE))

plot_learning_curve_for(lin_model,   X, y, "Learning Curve — Linear Regression (OLS)")
plot_learning_curve_for(ridge_model, X, y, "Learning Curve — Ridge (alpha=1.0)")
plot_learning_curve_for(tree_model,  X, y, "Learning Curve — Decision Tree (defaults)")

# 2) Validation curves for model complexity
# Ridge: alpha sweep
alpha_range = np.logspace(-3, 3, 15)
plot_validation_curve_for(
    "Ridge",
    Ridge(random_state=RANDOM_STATE),
    param_name="alpha",
    param_range=alpha_range,
    X=X, y=y
)

# Decision Tree: max_depth sweep
depth_range = np.arange(1, 31)  # 1..30
plot_validation_curve_for(
    "Decision Tree",
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_name="max_depth",
    param_range=depth_range,
    X=X, y=y
)

# (Optional) Quick numeric snapshot: train vs CV log-RMSE via cross_validate
for name, est in [
    ("OLS", LinearRegression()),
    ("Ridge", Ridge(random_state=RANDOM_STATE)),
    ("Tree", DecisionTreeRegressor(random_state=RANDOM_STATE)),
]:
    cv = cross_validate(
        wrap_model(est), X, y, cv=kf, scoring=log_rmse_scorer,
        return_train_score=True, n_jobs=None
    )
    train_score = -cv["train_score"].mean()
    val_score   = -cv["test_score"].mean()
    print(f"{name:>5} | train log-RMSE: {train_score:.4f} | CV log-RMSE: {val_score:.4f}")


In [ ]:
# ===== Robust feature analysis (fit on preprocessed matrix directly) =====

def _prep_matrix(X, y):
    """Fit preprocess on X, return transformed X (dense), log-transformed y, and feature names."""
    prep = preprocess.fit(X)                      # use your existing ColumnTransformer
    Xmat = prep.transform(X)
    # Ensure dense for plotting/inspection
    if hasattr(Xmat, "toarray"):
        Xmat = Xmat.toarray()
    y_log = np.log1p(y)                           # match our modeling metric
    feat_names = prep.get_feature_names_out()
    return prep, Xmat, y_log, feat_names

def top_linear_coeffs(estimator, X, y, k=15, title=None, return_df=False):
    """
    Fit a linear model (OLS or Ridge) on preprocessed X and log1p(y),
    then show top-|coef| features.
    """
    prep, Xmat, y_log, feat_names = _prep_matrix(X, y)
    estimator.fit(Xmat, y_log)

    if not hasattr(estimator, "coef_"):
        raise AttributeError(f"{type(estimator).__name__} did not expose coef_ after fitting.")

    coefs = np.asarray(estimator.coef_).ravel()
    coef_df = pd.DataFrame({"Feature": feat_names, "Coefficient": coefs})
    coef_df["AbsCoef"] = coef_df["Coefficient"].abs()
    top = coef_df.nlargest(k, "AbsCoef").copy()

    print(f"\nTop {k} |coefficients| — {type(estimator).__name__}")
    display(top[["Feature", "Coefficient", "AbsCoef"]].sort_values("AbsCoef", ascending=False))

    # Plot
    plt.figure(figsize=(8, max(4, int(k*0.45))))
    plt.barh(top["Feature"][::-1], top["Coefficient"][::-1])
    plt.xlabel("Coefficient (effect on log price)")
    plt.title(title or f"Top {k} coefficients — {type(estimator).__name__}")
    plt.tight_layout()
    plt.show()

    return top if return_df else None

def top_tree_importances(estimator=None, X=None, y=None, k=15, title=None, return_df=False):
    """
    Fit a DecisionTreeRegressor (or use best from GridSearchCV if available),
    trained on preprocessed X and log1p(y), then show top feature importances.
    """
    # try to use the tuned best tree from GridSearchCV if present
    try:
        fitted = grid.best_estimator_
        prep = fitted.regressor.named_steps["prep"]
        est  = fitted.regressor.named_steps["est"]
        feat_names = prep.get_feature_names_out()
        importances = np.asarray(est.feature_importances_, dtype=float)
    except Exception:
        # fallback: fit a fresh tree on preprocessed matrix directly
        if estimator is None:
            estimator = DecisionTreeRegressor(random_state=RANDOM_STATE)
        prep, Xmat, y_log, feat_names = _prep_matrix(X, y)
        estimator.fit(Xmat, y_log)
        importances = np.asarray(estimator.feature_importances_, dtype=float)

    imp_df = pd.DataFrame({"Feature": feat_names, "Importance": importances})
    imp_df = imp_df[imp_df["Importance"] > 0]
    top = imp_df.nlargest(k, "Importance").copy()

    tree_name = "DecisionTree (best from Grid)" if 'fitted' in locals() else type(estimator).__name__
    print(f"\nTop {k} importances — {tree_name}")
    display(top)

    # Plot
    plt.figure(figsize=(8, max(4, int(k*0.45))))
    plt.barh(top["Feature"][::-1], top["Importance"][::-1])
    plt.xlabel("Importance (total MSE decrease)")
    plt.title(title or f"Top {k} importances — {tree_name}")
    plt.tight_layout()
    plt.show()

    return top if return_df else None

# ==== Run analyses ====
top_linear_coeffs(LinearRegression(), X, y, k=15, title="Top 15 Coefficients — OLS")
top_linear_coeffs(Ridge(random_state=RANDOM_STATE), X, y, k=15, title="Top 15 Coefficients — Ridge")
top_tree_importances(None, X=X, y=y, k=15, title="Top 15 Importances — Decision Tree")

3. **Analyze the most significant variables for each algorithm.**

All models were trained to predict log(SalePrice), so each coefficient or importance describes how much the log of the price changes with a one-unit change in that feature, holding others constant.
Positive values indicate that the feature influences the model by increasing price, while negative values indicate that the feature influences the model by decreasing price.

A. OLS {Ordinary Least Squares} (Unregularized Linear Regression)

In the OLS model, the top important features are related with the roof material and the age of the house. As suspected, the older the house, the less value it has. On the other hand, the more recent a house was sold, the more value it has. As for roof top material, it seems that people prefer wood over clay. Wood may seem more premium, or is just a better material against the weather of the location, and clay might be associated with older, obsolete designs of houses.

B. Ridge Regression (Regularized Linear Model)

Ridge Regression generally has the same important features as OLS, the only difference is that they are not as influencial, a consequence of the regularization of the model.

C. Decision Tree Regressor

Although our decision tree is overfitting, we can still learn which features were important. Top-level splits still tell us genuine patterns. They are chosen because they create the biggest reduction in error across the whole dataset. While lower-level splits (deep in the tree) are where overfitting shows up.

Unlike linear models, the tree finds non-linear interactions and thresholds. The most influential feature was the Exterior Quality.

4. **Analyze the types of errors the models make.**

A. OLS {Ordinary Least Squares} (Unregularized Linear Regression)

OLS has no hyperparameters to tune, making it suscpetible to overfitting. However, since out task was too challenging for it, It did not overfit and generalized the data well.

B. Ridge Regression (Regularized Linear Model)

Ridge Regression has the hyperparameter "α", which controls how strong do we want our regularization impact to be. In cases were the linear model does overfit, we can increase the α hyperparameter to regularize the model. However, since even the OLS model did not overfit, the α hyperparameter was kept at one.

C. Decision Tree Regressor

A decision tree has 4 main hyperparameters: max_depth, min_samples_split, min_samples_leaf, max_leaf_nodes. All these hyperparameters can control model's performance in overfitting. In our case, we used the default parameters which have no limit on the maximum depth of the tree, which is most likely what caused our overfitting as the graph suggests. Having a deep tree means that the model is learning too many details and not generalizing the features, thus overfitting.

Regularization for the linear model is not doing anything, since the model does not overfit from the beginning. The decision tree is doing the worst since it overfits alot.

Regularization (Ridge) adds minimal benefit because the OLS model was already generalizing well (low variance, modest feature set).

Decision tree shows strong overfitting. It needs regularization (max_depth, min_samples_leaf) or an ensemble method (Random Forest, Gradient Boosting) to improve.

# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


linear model using RandomizedSearchCV

In [ ]:
# ===== Ridge + RandomizedSearchCV on log target (warning-free) =====
from sklearn.linear_model import Ridge
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import make_scorer, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from scipy.stats import loguniform
import numpy as np
import pandas as pd

# --- 5) Model on log target; evaluate with log-RMSE ---
def log_rmse(y_true, y_pred):
    """Compute RMSE on log-transformed targets."""
    return root_mean_squared_error(np.log1p(y_true), np.log1p(y_pred))

# Custom scorer for Kaggle metric
log_rmse_scorer = make_scorer(log_rmse, greater_is_better=False)

# Base model (Ridge defaults)
base_model = Ridge(random_state=RANDOM_STATE, max_iter=10000)

# Wrap to learn on log1p(SalePrice)
full_model = TransformedTargetRegressor(
    regressor=Pipeline([("prep", preprocess), ("ridge", base_model)]),
    func=np.log1p,
    inverse_func=np.expm1,
)

# CV splitter
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- RandomizedSearch over Ridge alpha (log-uniform) ---
param_distributions = {
    "regressor__ridge__alpha": loguniform(1e-3, 1e3)
}

search = RandomizedSearchCV(
    estimator=full_model,
    param_distributions=param_distributions,
    n_iter=40,
    scoring=log_rmse_scorer,
    cv=kf,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=0
)

search.fit(X, y)

best_cv = -search.best_score_  # Flip sign for positive log-RMSE
print(f"Best CV log-RMSE: {best_cv:.5f}")
print("Best params:", search.best_params_)

# --- 6) Fit on full train (already refit=True) & predict test ---
best_model = search.best_estimator_
test_pred = best_model.predict(X_test)
test_pred = np.maximum(test_pred, 0)  # Ensure no negatives

# --- 7) Save submission ---
sub = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv", index_col="Id")
sub["SalePrice"] = test_pred
sub.to_csv("/kaggle/working/submission.csv")
print("Wrote /kaggle/working/submission.csv")

# --- 8) Traceability ---
print("\nDropped due to missing values:", len(na_cols_union), na_cols_union[:10], "..." if len(na_cols_union) > 10 else "")
print("Dropped due to outliers:", len(outlier_cols), outlier_cols[:10], "..." if len(outlier_cols) > 10 else "")
print("Remaining shape (train):", X.shape, "| (test):", X_test.shape)


The randomizedsearchCV has found that the best parameter for alpha is 3.58, which has slightly improved the error.

**Decision tree using GridSearchCV.**

In [ ]:
# ===== Decision Tree + GridSearchCV on log target (warning-free) =====
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import make_scorer, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
import numpy as np
import pandas as pd

# --- Kaggle-style scorer: RMSE on log1p scale (lower is better) ---
def log_rmse(y_true, y_pred):
    return root_mean_squared_error(np.log1p(y_true), np.log1p(y_pred))

log_rmse_scorer = make_scorer(log_rmse, greater_is_better=False)

# Base tree
base_model = DecisionTreeRegressor(random_state=RANDOM_STATE)

# Wrap to learn on log1p(SalePrice)
full_model = TransformedTargetRegressor(
    regressor=Pipeline([("prep", preprocess), ("tree", base_model)]),
    func=np.log1p,
    inverse_func=np.expm1,
)

# 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Grid over key tree complexity controls ---
param_grid = {
    "regressor__tree__max_depth": [3, 5, 7, 9, 12, 15, None],
    "regressor__tree__min_samples_leaf": [1, 2, 5, 10, 20],
    "regressor__tree__min_samples_split": [2, 5, 10, 20],
    # Optional knobs (uncomment to explore more; increases grid size):
    # "regressor__tree__max_features": [None, "sqrt", "log2"],
    # "regressor__tree__max_leaf_nodes": [None, 50, 100, 200],
}

grid = GridSearchCV(
    estimator=full_model,
    param_grid=param_grid,
    scoring=log_rmse_scorer,
    cv=kf,
    n_jobs=-1,
    refit=True,
    verbose=1
)

grid.fit(X, y)

best_cv = -grid.best_score_  # flip sign for positive log-RMSE
print(f"\nBest CV log-RMSE: {best_cv:.5f}")
print("Best params:", grid.best_params_)

# --- Fit on full train (already refit=True) & predict test ---
best_model = grid.best_estimator_
test_pred = best_model.predict(X_test)
test_pred = np.maximum(test_pred, 0)  # ensure non-negative

# --- Save submission ---
sub = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv", index_col="Id")
sub["SalePrice"] = test_pred
sub.to_csv("/kaggle/working/submission.csv")
print("Wrote /kaggle/working/submission.csv")

# --- Traceability ---
print("\nDropped due to missing values:", len(na_cols_union), na_cols_union[:10], "..." if len(na_cols_union) > 10 else "")
print("Dropped due to outliers:", len(outlier_cols), outlier_cols[:10], "..." if len(outlier_cols) > 10 else "")
print("Remaining shape (train):", X.shape, "| (test):", X_test.shape)


In [ ]:
# ===== Learning curve for the best Decision Tree (from GridSearchCV) =====
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve, KFold
from sklearn.metrics import make_scorer, root_mean_squared_error

# --- Kaggle-style scorer: RMSE on log1p scale (lower is better) ---
def log_rmse(y_true, y_pred):
    return root_mean_squared_error(np.log1p(y_true), np.log1p(y_pred))

log_rmse_scorer = make_scorer(log_rmse, greater_is_better=False)

# Ensure we have a fitted grid with best_estimator_
try:
    best_model = grid.best_estimator_
except NameError:
    raise RuntimeError("GridSearchCV result `grid` not found. Run the Decision Tree + GridSearchCV cell first.")

# Learning curve settings
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_sizes = np.linspace(0.1, 1.0, 8)

# Compute learning curves
sizes, train_scores, val_scores = learning_curve(
    estimator=best_model,
    X=X,
    y=y,
    cv=kf,
    scoring=log_rmse_scorer,
    train_sizes=train_sizes,
    shuffle=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# learning_curve returns negative scores since greater_is_better=False → flip sign
train_mean = -train_scores.mean(axis=1)
train_std  =  train_scores.std(axis=1)
val_mean   = -val_scores.mean(axis=1)
val_std    =  val_scores.std(axis=1)

# Plot
plt.figure(figsize=(7,5))
plt.title("Learning Curve — Decision Tree (best from GridSearchCV)")
plt.xlabel("Training examples")
plt.ylabel("log-RMSE")
plt.grid(True)

plt.plot(sizes, train_mean, label="Training score")
plt.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)

plt.plot(sizes, val_mean, label="CV score")
plt.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.15)

plt.legend()
plt.show()

print(f"Final CV log-RMSE at full data: {val_mean[-1]:.5f}  |  Train log-RMSE: {train_mean[-1]:.5f}")


The new fine-tuned decision tree performs better than the previous one. However, the best model out of all was the fine-tuned regularized model. It appears that even with fine-tuning, this task is too simple for the decision tree. Perhaps if we added an ensemble method, we could get better results.

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

1. We developed a predictive model that estimates home sale prices in Ames, Iowa, using property characteristics such as size, quality, and age.
After testing several algorithms, a regularized linear regression (Ridge) model provided the most accurate and consistent price predictions, achieving about 20% average prediction error while avoiding overfitting.
This approach can help better understand what factors drive property values and improve pricing and investment decisions. The most influential factors we discovered were house age, roof top material, and exterior quality.

2. The solution used the Ames Housing dataset (79 features) with preprocessing steps that removed features containing missing or outlier values, added engineered variables (HouseAge, YearsSinceRemodel), and applied one-hot encoding for categorical fields.
Multiple models—OLS, Ridge, and Decision Tree regressors—were evaluated using 5-fold cross-validated log-RMSE.
A Ridge Regression tuned via RandomizedSearchCV achieved the best generalization (CV log-RMSE ≈ 0.20), outperforming an optimized Decision Tree (≈ 0.24) by maintaining lower variance on unseen data.
Future improvement paths include ensemble methods (Random Forests, Gradient Boosting) to capture non-linear interactions while preserving regularization, or better feature selection/ feature engineering.

# 9. Submit your predictions to the competition

In [ ]:
test_predictions = best_model.predict(X_test)

In [ ]:
submission_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    )
    .assign(SalePrice=test_predictions)
)

# Save to working directory
submission_df.to_csv("submission.csv", index=True)
print("✅ Submission file 'submission.csv' created successfully!")


In [ ]:
%%bash

cat submission.csv | head -n 5